# 00 – Gerar metadados automaticamente

Este notebook lê os arquivos CSV da pasta `raw/` e gera os arquivos JSON de metadados correspondentes, seguindo o schema definido em `metadata_schema.json`.

Para cada CSV, você será solicitado a informar:
 - Nome do dataset
 - Fonte
 - Descrição
 - Observações
 
Os valores padrão são sugeridos com base no nome do arquivo.


In [1]:
import pandas as pd
import json
from pathlib import Path
from datetime import date

# Diretórios
RAW_DIR = Path("../raw")
SCHEMA_PATH = Path("../metadata_schema.json")

# Carregar schema (opcional, apenas para referência)
if SCHEMA_PATH.exists():
    with open(SCHEMA_PATH, 'r', encoding='utf-8') as f:
        schema = json.load(f)
    print("Schema carregado.")

Schema carregado.


In [2]:
def gerar_metadados(csv_path, nome_dataset, fonte, descricao, observacoes):
    """
    Gera um dicionário de metadados para o dataset.
    Os campos principais são extraídos das primeiras 5 linhas do CSV.
    """
    # Ler amostra para obter os nomes das colunas
    df_sample = pd.read_csv(csv_path, nrows=5)
    campos_principais = list(df_sample.columns)
    
    metadados = {
        "nome_dataset": nome_dataset,
        "fonte": fonte,
        "descricao": descricao,
        "campos_principais": campos_principais,
        "data_criacao": date.today().isoformat(),
        "observacoes": observacoes
    }
    return metadados

In [3]:
# Listar todos os CSVs na pasta raw
csv_files = list(RAW_DIR.glob("*.csv"))

if not csv_files:
    print("Nenhum arquivo CSV encontrado em raw/")
else:
    print(f"Encontrados {len(csv_files)} arquivos CSV.")


Encontrados 2 arquivos CSV.


In [4]:
for csv_path in csv_files:
    print(f"\n--- {csv_path.name} ---")
    
    # Definir valores padrão baseados no nome do arquivo
    if "historico" in csv_path.stem:
        default_nome = "Histórico de Medalhas Olímpicas por País (1896-2022)"
        default_fonte = "Base dos Dados (basedosdados.org) – tabela medalhas"
        default_desc = "Quadro de medalhas por país e edição olímpica, contendo ouro, prata, bronze e total."
        default_obs = "Inclui edições de verão e inverno. Cada linha representa um país em uma edição."
    elif "paris" in csv_path.stem:
        default_nome = "Medalhistas de Paris 2024"
        default_fonte = "Kaggle – Paris 2024 Olympic Summer Games (piterfm/paris-2024-olympic-summer-games) – medallists.csv"
        default_desc = "Lista de todas as medalhas concedidas nos Jogos de Paris 2024, com detalhes do atleta, disciplina, país, gênero."
        default_obs = "Cada linha representa uma medalha (ouro, prata, bronze) por atleta ou equipe."
    else:
        default_nome = csv_path.stem.replace('_', ' ').title()
        default_fonte = ""
        default_desc = ""
        default_obs = ""
    
    # Solicitar informações ao usuário
    nome = input(f"Nome do dataset [{default_nome}]: ").strip()
    if not nome:
        nome = default_nome
    
    fonte = input(f"Fonte [{default_fonte}]: ").strip()
    if not fonte:
        fonte = default_fonte
    
    desc = input(f"Descrição [{default_desc}]: ").strip()
    if not desc:
        desc = default_desc
    
    obs = input(f"Observações [{default_obs}]: ").strip()
    if not obs:
        obs = default_obs
    
    metadados = gerar_metadados(csv_path, nome, fonte, desc, obs)
    
    # Salvar JSON
    json_path = csv_path.with_suffix(".json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(metadados, f, indent=2, ensure_ascii=False)
    
    print(f"Metadados salvos em {json_path}")

print("\n✅ Todos os metadados foram gerados.")


--- olympics_historico.csv ---
Metadados salvos em ../raw/olympics_historico.json

--- olympics_paris2024.csv ---
Metadados salvos em ../raw/olympics_paris2024.json

✅ Todos os metadados foram gerados.
